# 11 · Hugging Face corpora — Lucius-Morningstar

Navigate and explore the datasets the mailroom family publishes on
[Lucius-Morningstar](https://huggingface.co/Lucius-Morningstar): CUAD
contracts, LegalBench, docclass packs, Enron correspondence, and CMS
DE-SynPUF insurance claims — each mapped onto a mailroom document class.

**What you'll see:** the org catalog, first-row previews, substring search,
an equality filter, and one Hub row fed into the real pipeline (mock LLM).

**Honesty label:** default cells are OFFLINE. They read a committed Dataset
Viewer snapshot under `notebooks/fixtures/huggingface/` (dated in
`catalog.json`). Nothing below talks to the Hub unless you set
`MAILROOM_HF_LIVE=1` and run the marker-gated live cell at the bottom.
`legalbench-full` on the Hub is currently a stub viewer (placeholder rows);
notebook 12 is the real LegalBench suite.

Companion: `notebooks/huggingface_lab.py`.


## Setup


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "notebooks" / "pipeline_lab.py").exists(), (
    f"llm-mailroom repo root not found above {Path.cwd()}"
)
sys.path.insert(0, str(ROOT / "notebooks"))
sys.path.insert(0, str(ROOT / "src"))

import huggingface_lab as hf
import pipeline_lab as lab
lab.quiet_logs()
print("snapshot date:", hf.catalog()["snapshot_date"])
print("source:", hf.catalog()["source"] if "source" in hf.catalog() else "offline-snapshot")
print("live requested:", hf.live_requested())


snapshot date: 2026-08-25
source: offline-snapshot
live requested: False


## The catalog

Seven datasets. `mailroom_classes` is the wiring back to `taxonomy.yaml`,
not a Hub tag — it is how this pipeline consumes the published surface.


In [2]:
cat = hf.catalog()
hf.show_catalog(cat["datasets"])
print()
print("org:", cat["org_url"])


dataset                                              rows  classes                  role
------------------------------------------------------------------------------------------------------------------------
docclass-merged                                     2,420  contract,corporate_record,correspondence,compliance_filing,due_diligence,court_opinion Merged document-classification training pack (SE
docclass-pilot                                        276  contract,corporate_record,correspondence,compliance_filing Smaller pilot slice of the same docclass pack us
enron-correspondence-dedup                        247,523  correspondence           Deduplicated Enron mail — correspondence special
cms-desynpuf-insurance-claims                         400  insurance_claim          CMS DE-SynPUF synthetic Medicare claims (insuran
mailroom-cuad-contracts                               546  contract                 Atticus CUAD contract pages as an image folder (
mailroom-cuad-contracts-full 

## Preview — insurance claims (CMS DE-SynPUF)

Synthetic Medicare claims, labeled `insurance_claim`. No real PHI.


In [3]:
claims = hf.preview("Lucius-Morningstar/cms-desynpuf-insurance-claims", length=3)
hf.show_rows(claims, text_chars=140)
print("features:", claims.get("features"))


Lucius-Morningstar/cms-desynpuf-insurance-claims  source=offline-snapshot  n=3
  [1]
      doc_text: ========================================================================== MEDICARE SUMMARY NOTICE -- PHYSICIAN/SUPPLIER CLAIM (Part B) Noti…
      expected: insurance_claim
      expected_subclass: carrier
      filename: carrier:887023387197025.txt
      metadata: {"claim_subtype": "carrier", "diagnosis_codes": ["5899"], "ground_truth": {"adjuster": null, "claim_number": "887023387197025", "claim_type"…
      prompt: 
  [2]
      doc_text: ========================================================================== MEDICARE SUMMARY NOTICE -- PHYSICIAN/SUPPLIER CLAIM (Part B) Noti…
      expected: insurance_claim
      expected_subclass: carrier
      filename: carrier:887743385537772.txt
      metadata: {"claim_subtype": "carrier", "diagnosis_codes": ["2334", "78701", "7245", "78833"], "ground_truth": {"adjuster": null, "claim_number": "8877…
      prompt: 
  [3]
      doc_text: =======

## Preview — Enron correspondence (deduped)


In [4]:
enron = hf.preview("Lucius-Morningstar/enron-correspondence-dedup", length=3)
hf.show_rows(enron, text_chars=120)


Lucius-Morningstar/enron-correspondence-dedup  source=offline-snapshot  n=3
  [1]
      filename: allen-p/_sent_mail/1.
      subject: 
      text: Here is our forecast
      split: train
      metadata: {"source": "cmu_enron_maildir", "license": "Enron corpus \u2014 released for research use", "built_by": "llm-entity-extr…
  [2]
      filename: allen-p/_sent_mail/10.
      subject: Re:
      text: Traveling to have a business meeting takes the fun out of the trip. Especially if you have to prepare a presentation. I …
      split: train
      metadata: {"source": "cmu_enron_maildir", "license": "Enron corpus \u2014 released for research use", "built_by": "llm-entity-extr…
  [3]
      filename: allen-p/_sent_mail/100.
      subject: Re: test
      text: test successful. way to go!!!
      split: train
      metadata: {"source": "cmu_enron_maildir", "license": "Enron corpus \u2014 released for research use", "built_by": "llm-entity-extr…


## Preview — CUAD full contracts (clause labels in `expected`)


In [5]:
cuad = hf.preview("Lucius-Morningstar/mailroom-cuad-contracts-full", length=2)
hf.show_rows(cuad, text_chars=140)


Lucius-Morningstar/mailroom-cuad-contracts-full  source=offline-snapshot  n=2
  [1]
      id: cuad-Monsanto Company - SECOND A_R EXCLUSIVE AGENCY AND MARKETING AGREEMENT 
      input: {"doc_text":"Exhibit 10.14(a)\n\nSECOND AMENDED AND RESTATED  EXCLUSIVE AGENCY AND  MARKETING AGREEM
      expected: {"clause_count":41,"clause_labels":[{"answer":"SECOND AMENDED AND RESTATED EXCLUSIVE AGENCY AND MARK
      metadata: {"applicable_categories":["Affiliate License-Licensee","Agreement Date","Anti-Assignment","Audit Rig
      tags: None
      created: 2026-08-10T04:41:04.395Z
  [2]
      id: cuad-MetLife, Inc. - Remarketing Agreement
      input: {"doc_text":"Exhibit 99.1\n\nEXECUTION VERSION\n\nMETLIFE, INC.\n\nSeries E Senior Component Debentu
      expected: {"clause_count":41,"clause_labels":[{"answer":"Remarketing Agreement","answer_start":364,"question":
      metadata: {"applicable_categories":["Affiliate License-Licensee","Agreement Date","Anti-Assignment","Audit Rig
      tags: None


## Search the snapshot

Offline search is a substring over the committed first-row window — not the
full 247k Enron rows. The live cell at the bottom uses Dataset Viewer `/search`.


In [6]:
hits = hf.search("Lucius-Morningstar/enron-correspondence-dedup", "forecast")
print("query=forecast  hits=", len(hits["rows"]), " source=", hits["source"])
hf.show_rows(hits, text_chars=100)


query=forecast  hits= 1  source= offline-snapshot-substring
Lucius-Morningstar/enron-correspondence-dedup  source=offline-snapshot-substring  n=1
  [1]
      filename: allen-p/_sent_mail/1.
      subject: 
      text: Here is our forecast
      split: train
      metadata: {"source": "cmu_enron_maildir", "license": "Enron corpus \u2014 released for research use", "built_b…


## Filter — insurance rows labeled `insurance_claim`


In [7]:
filt = hf.filter_rows(
    "Lucius-Morningstar/cms-desynpuf-insurance-claims",
    where="expected=insurance_claim",
)
print("hits:", len(filt["rows"]), "source:", filt["source"])
print("labels:", sorted({r.get("expected") for r in filt["rows"]}))


hits: 8 source: offline-snapshot-equality
labels: ['insurance_claim']


## Feed a Hub row into the pipeline

Take the first DE-SynPUF claim's `doc_text`, run it as an `insurance_claim`
through the real graph (mock specialist). The catalog join in
`dataset_browser` is the local-pilot analogue of this.


In [8]:
row = claims["rows"][0]
text = hf.row_to_doc_text(row)
print("filename:", row.get("filename"), " chars:", len(text))
env = lab.open_sandbox()
lab.script_all_specialists(env["client"])
run = lab.run_document(
    env, text[:4000], filename="hf_claim.txt",
    classification=lab.CLASSIFY_INSURANCE_HIGH,
    extraction=lab.INSURANCE_CLAIM_EXTRACTION,
)
print("path:", " → ".join(lab.path_of(run["steps"])))
print("stage:", run["final"].get("stage"), " type:", run["final"].get("doc_type"))
lab.close_sandbox(env)


filename: carrier:887023387197025.txt  chars: 501


path: ingest-document → classify-document → extract-fields → compile-report → write-catalog → archive-document
stage: archived  type: insurance_claim


## Docclass + CUAD + LegalBench at a glance

| dataset | mailroom use |
|---|---|
| `docclass-merged` / `docclass-pilot` | sorter training / classification experiments |
| `mailroom-cuad-contracts` | vision surface (page images) |
| `mailroom-cuad-contracts-full` | contract texts + CUAD clause labels |
| `legalbench-full` | Hub stub — use notebook 12 |
| `enron-correspondence-dedup` | correspondence specialist |
| `cms-desynpuf-insurance-claims` | insurance_claim specialist |

Local pilot samples (30 rows, no Hub) still live in `dataset_browser.ipynb`.


## Live Hub refresh (opt-in)

<!-- NB-OPT-IN-NETWORK: Dataset Viewer / Hub API; skipped unless MAILROOM_HF_LIVE=1 -->

Set `MAILROOM_HF_LIVE=1` (optional `HF_TOKEN` for higher rate limits) and
re-run to hit `https://datasets-server.huggingface.co` for a fresh catalog
and a live search. Default execution never takes this branch.


In [9]:
import os
if os.environ.get("MAILROOM_HF_LIVE", "").strip().lower() in ("1", "true", "yes", "on"):
    live_cat = hf.catalog(live=True)
    print("LIVE catalog source:", live_cat.get("source"), "n=", len(live_cat["datasets"]))
    live_hits = hf.search("Lucius-Morningstar/enron-correspondence-dedup", "forecast", live=True)
    print("LIVE search hits:", live_hits.get("num_rows_total"), "source:", live_hits.get("source"))
else:
    print("live cell skipped (MAILROOM_HF_LIVE not set) — offline snapshot used above.")


live cell skipped (MAILROOM_HF_LIVE not set) — offline snapshot used above.


## Where to go next

- **12 legalbench** — the in-repo eval suite (binary QA + family classification)
- **13 vision_ingestion** — page images, the other CUAD surface
- **dataset_browser** — the 30 local pilot samples + catalog overlay
